In [1]:
!pip install -U torch torchvision torchaudio
!pip install -U transformers accelerate peft bitsandbytes trl datasets qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 67.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 84.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 63.8 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 81.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 60.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 63.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 65.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/28

In [1]:
import torch
import gc

# Delete the model and processor variables if they exist
try:
    del model
    del processor
except NameError:
    pass

# Force Python's garbage collector to run
gc.collect()

# Empty the PyTorch CUDA cache
torch.cuda.empty_cache()

In [1]:
import os

# 1. Route all Hugging Face downloads to your spacious persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# 2. Keep the fix for the Xet network timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Hugging Face cache set to: {os.environ['HF_HOME']}")

Hugging Face cache set to: /workspace/huggingface_cache


In [3]:
!pip install thefuzz


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [6]:
!pip install flash-attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 40.2 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
anceled

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
ERROR: Operation cancelled by user


In [4]:
import os
import json
import torch
import pandas as pd
import numpy as np
from rapidfuzz import fuzz
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
from qwen_vl_utils import process_vision_info

# --- Configuration Paths ---
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
ADAPTER_PATH = "/workspace/checkpoint-486" # UPDATE THIS

TEST_JSON_PATH = "extracted_chart_specs_test_300.json"
TEST_IMG_DIR = "./test_images_300"
PREDICTIONS_OUTPUT = "predicted_specs_test.json"
METRICS_CSV = "stage1_evaluation_results.csv"

# Load Ground Truth
with open(TEST_JSON_PATH, 'r', encoding="utf-8") as f:
    ground_truth_data = json.load(f)

import json
import numpy as np
try:
    from thefuzz import fuzz
except ImportError:
    from fuzzywuzzy import fuzz

def _parse_mixed_value(val):
    """Safely handles integers, floats, and scientific notation strings."""
    if isinstance(val, (int, float)) and not isinstance(val, bool):
        return float(val), True
    if isinstance(val, str):
        clean_v = val.replace('−', '-').replace(',', '').strip()
        try:
            return float(clean_v), True
        except ValueError:
            return str(val).strip(), False
    return str(val), False

def _clean_math_equation(eq_str):
    """Removes the assignment variables to isolate the math."""
    if not isinstance(eq_str, str):
        return ""
    if '=' in eq_str:
        eq_str = eq_str.split('=', 1)[-1]
    return eq_str.strip().replace(" ", "")

def evaluate_json_accuracy(reference_json, model_json):
    results = {
        "valid_json": False,
        "schema_score": 0.0,
        "topology_score": 0.0,
        "domain_accuracy": 0.0,
        "numerical_accuracy": 0.0,
        "categorical_accuracy": 0.0,
        "math_accuracy": None,
        "semantic_score": 0.0
    }
    
    try:
        ref = json.loads(reference_json)
        pred = json.loads(model_json)
        results["valid_json"] = True
    except Exception:
        return results

    # ==========================================
    # 1. SCHEMA & TRUNCATION FLAGS
    # ==========================================
    ref_keys = set(ref.keys())
    pred_keys = set(pred.keys())
    if ref_keys:
        results["schema_score"] = len(ref_keys & pred_keys) / len(ref_keys)

    if ref.get("is_truncated_for_context"):
        if pred.get("is_truncated_for_context"):
            results["schema_score"] = min(1.0, results["schema_score"] + 0.1)
        else:
            results["schema_score"] -= 0.1

    # ==========================================
    # 2. TOPOLOGY ALIGNMENT
    # ==========================================
    ref_topo = ref.get("topo", {})
    pred_topo = pred.get("topo", {})
    topo_matches = 0
    if ref_topo.get("type") == pred_topo.get("type"): topo_matches += 0.5
    if ref_topo.get("sub") == pred_topo.get("sub"): topo_matches += 0.5
    results["topology_score"] = topo_matches

    # ==========================================
    # 3. DOMAIN & AXES CHECK
    # ==========================================
    ref_axes = ref.get("axes", [])
    pred_axes = pred.get("axes", [])
    dom_scores = []
    
    for r_ax, p_ax in zip(ref_axes, pred_axes):
        r_dom, p_dom = r_ax.get("dom", []), p_ax.get("dom", [])
        if not r_dom or not p_dom: continue
        
        for r_val, p_val in zip(r_dom, p_dom):
            r_v, r_is_num = _parse_mixed_value(r_val)
            p_v, p_is_num = _parse_mixed_value(p_val)
            
            if r_is_num and p_is_num:
                margin = max(abs(r_v) * 0.05, 1e-5)
                dom_scores.append(1.0 if abs(r_v - p_v) <= margin else 0.0)
            elif not r_is_num and not p_is_num:
                dom_scores.append(1.0 if str(r_v).lower() == str(p_v).lower() else 0.0)
            else:
                dom_scores.append(0.0) # Type mismatch penalty
                
    if dom_scores:
        length_penalty = min(1.0, len(pred_axes) / max(1, len(ref_axes)))
        results["domain_accuracy"] = np.mean(dom_scores) * length_penalty

    # ==========================================
    # 4. GENERATIVE MATH CHECK
    # ==========================================
    ref_math = ref.get("math", [])
    pred_math = pred.get("math", [])
    
    if ref_math:
        r_eqs = [_clean_math_equation(eq) for eq in ref_math]
        p_eqs = [_clean_math_equation(eq) for eq in pred_math]
        
        math_scores = []
        for r_eq in r_eqs:
            if not p_eqs:
                math_scores.append(0.0)
                continue
            best_score = max([fuzz.ratio(r_eq, p_eq) for p_eq in p_eqs])
            math_scores.append(best_score / 100.0)
            
        results["math_accuracy"] = np.mean(math_scores)
    elif pred_math:
        # Penalize hallucinating math equations when the chart doesn't require them
        results["math_accuracy"] = 0.0

    # ==========================================
    # 5. SERIES DATA CHECK 
    # ==========================================
    ref_ser = ref.get("ser", [])
    pred_ser = pred.get("ser", [])
    
    num_errors = []
    cat_matches = []
    
    for i in range(min(len(ref_ser), len(pred_ser))):
        r_data, p_data = ref_ser[i].get("data", []), pred_ser[i].get("data", [])
        
        for j in range(min(len(r_data), len(p_data))):
            r_pt, p_pt = r_data[j], p_data[j]
            if not isinstance(r_pt, (list, tuple)) or not isinstance(p_pt, (list, tuple)): continue
            
            for k in range(min(len(r_pt), len(p_pt))):
                r_v, r_is_num = _parse_mixed_value(r_pt[k])
                p_v, p_is_num = _parse_mixed_value(p_pt[k])
                
                if r_is_num and p_is_num:
                    denominator = abs(r_v) if abs(r_v) > 1e-9 else 1e-9
                    error_ratio = min(1.0, abs(r_v - p_v) / denominator) 
                    num_errors.append(error_ratio)
                elif not r_is_num and not p_is_num:
                    cat_matches.append(1.0 if str(r_v).lower() == str(p_v).lower() else 0.0)
                else:
                    # FIX: Apply full penalty if expected numeric but got categorical (or vice versa)
                    if r_is_num:
                        num_errors.append(1.0)
                    else:
                        cat_matches.append(0.0)

    # --- Safe Numerical Accuracy Aggregation ---
    r_total_pts = sum(len(s.get("data", [])) for s in ref_ser)
    p_total_pts = sum(len(s.get("data", [])) for s in pred_ser)

    if r_total_pts > 0 and num_errors:
        # FIX: Symmetric length penalty. Penalizes both dropping points AND hallucinating extra points.
        length_penalty = min(r_total_pts, p_total_pts) / max(1, max(r_total_pts, p_total_pts))
        base_acc = 1.0 - np.mean(num_errors)
        results["numerical_accuracy"] = max(0.0, base_acc * length_penalty)
    elif r_total_pts == 0 and ref_math:
        results["numerical_accuracy"] = results.get("math_accuracy", 1.0)

    if cat_matches:
        results["categorical_accuracy"] = np.mean(cat_matches)

    # ==========================================
    # 6. SEMANTIC & REASONING SCORE
    # ==========================================
    semantic_checks = []
    
    for i in range(min(len(ref_ser), len(pred_ser))):
        r_stats, p_stats = ref_ser[i].get("stats", {}), pred_ser[i].get("stats", {})
        r_trend, p_trend = ref_ser[i].get("trend", {}), pred_ser[i].get("trend", {})
        
        if r_trend.get("global_trend"):
            semantic_checks.append(1.0 if r_trend.get("global_trend") == p_trend.get("global_trend") else 0.0)
            
        r_max, r_is_num = _parse_mixed_value(r_stats.get("max_value"))
        p_max, p_is_num = _parse_mixed_value(p_stats.get("max_value"))
        if r_is_num and p_is_num:
            margin = max(abs(r_max) * 0.05, 1e-5)
            semantic_checks.append(1.0 if abs(r_max - p_max) <= margin else 0.0)

    # FIX: Check for both 'relation' and 'description' keys from the extractor
    ref_rels = " ".join([r.get("relation", r.get("description", "")) for r in ref.get("rel", [])])
    pred_rels = " ".join([r.get("relation", r.get("description", "")) for r in pred.get("rel", [])])
    
    if ref_rels:
        rel_score = fuzz.ratio(ref_rels, pred_rels) / 100.0
        semantic_checks.append(rel_score)

    if semantic_checks:
        results["semantic_score"] = np.mean(semantic_checks)

    return results

def extract_numbers(d):
    """Recursively find all floats/ints in the JSON"""
    nums = []
    if isinstance(d, dict):
        for v in d.values():
            nums.extend(extract_numbers(v))
    elif isinstance(d, list):
        for v in d:
            nums.extend(extract_numbers(v))
    elif isinstance(d, (int, float)) and not isinstance(d, bool):
        nums.append(float(d))
    return nums

In [ ]:
This is a very important update. When running long batch jobs on large datasets like ChartCheck or your TPM report test sets, you don't want a single network timeout or OOM (Out-Of-Memory) error to wipe out hours of progress.

The following script implements a "Resume & Save" logic by checking if PREDICTIONS_OUTPUT exists, loading its IDs, and skipping them. It also saves the JSON to disk after every single inference.

Updated Batch Inference Script with Resume Logic
Python
import json
import os
import torch
from peft import PeftModel
from transformers import AutoProcessor, AutoModelForImageTextToText
from qwen_vl_utils import process_vision_info

# ==========================================
# PHASE 1: INFERENCE & SAVING PREDICTIONS
# ==========================================
def run_inference_and_save():
    # 1. Load Model & Processor
    print(f"Loading Base Processor from {BASE_MODEL_ID}...")
    processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

    print("Loading Base Model into VRAM...")
    base_model = AutoModelForImageTextToText.from_pretrained(
        BASE_MODEL_ID,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa"
    )

    print(f"Attaching LoRA Adapter from {ADAPTER_PATH}...")
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

    print("Merging weights for ultra-fast inference...")
    model = model.merge_and_unload()
    model.eval()
    print("✅ Model successfully assembled in memory!\n")

    # 2. Setup Resume Logic
    predictions_record = []
    processed_ids = set()

    if os.path.exists(PREDICTIONS_OUTPUT):
        try:
            with open(PREDICTIONS_OUTPUT, 'r', encoding="utf-8") as f:
                predictions_record = json.load(f)
                processed_ids = {item["id"] for item in predictions_record}
            print(f"🔄 Resuming: Found {len(processed_ids)} existing predictions in {PREDICTIONS_OUTPUT}.")
        except json.JSONDecodeError:
            print(f"⚠️ Warning: {PREDICTIONS_OUTPUT} was corrupted or empty. Starting from scratch.")

    # 3. Load Ground Truth Data
    with open(TEST_JSON_PATH, 'r', encoding="utf-8") as f:
        ground_truth_data = json.load(f)

    # Filter out already processed items
    remaining_data = [item for item in ground_truth_data if item.get("id") not in processed_ids]
    print(f"📊 Total: {len(ground_truth_data)} | Already Done: {len(processed_ids)} | Remaining: {len(remaining_data)}\n")

    # 4. Main Inference Loop
    for index, gt_item in enumerate(remaining_data, start=1):
        chart_id = gt_item.get("id")
        ctype = gt_item.get("chart_type")
        complexity = gt_item.get("complexity")
        img_path = os.path.join(TEST_IMG_DIR, f"{chart_id}.png")
        
        print(f"Generating [{len(processed_ids) + index}/{len(ground_truth_data)}] - {ctype} (ID: {chart_id})")
        
        if not os.path.exists(img_path):
            print(f"  ⚠️ Image not found: {img_path}. Skipping.")
            continue

        messages = [
            {"role": "system", "content": "You are a chart data extraction specialist."},
            {"role": "user", "content": [
                {"type": "image", "image": img_path, "max_pixels": 768 * 768},
                {"type": "text", "text": "Extract the Extended ChartSpec JSON from this chart image."}
            ]}
        ]

        # Process Inputs
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        # Generate Output
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs, 
                max_new_tokens=4096,
                temperature=0.1,  
                do_sample=False
            )

        # Decode
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        prediction_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        # Update Record
        predictions_record.append({
            "id": chart_id,
            "chart_type": ctype,
            "complexity": complexity,
            "predicted_text": prediction_text
        })

        # --- THE SAVE-AFTER-EACH-INFERENCE FIX ---
        # We overwrite the file every time. Since it's JSON, this is safest for data integrity.
        with open(PREDICTIONS_OUTPUT, 'w', encoding="utf-8") as f:
            json.dump(predictions_record, f, indent=2, ensure_ascii=False)

    print(f"\n✅ All predictions successfully synchronized and saved to {PREDICTIONS_OUTPUT}")
    
    del model
    del base_model
    torch.cuda.empty_cache()

if __name__ == "__main__":
    run_inference_and_save()

Loading Base Processor from Qwen/Qwen2.5-VL-7B-Instruct...
Loading Base Model into VRAM...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Attaching LoRA Adapter from /workspace/checkpoint-486...
Merging weights for ultra-fast inference...


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Model successfully assembled in memory!

Generating [1/258] - Histogram (ID: 131740)
Generating [2/258] - Scatter Plot (ID: 149999)
Generating [3/258] - Line Chart (ID: 153147)
Generating [4/258] - Line Chart (ID: 149267)
Generating [5/258] - Line Chart (ID: 107744)
Generating [6/258] - Line Chart (ID: 90311)
Generating [7/258] - Area Chart (ID: 55210)
Generating [8/258] - Pie Chart (ID: 71634)
Generating [9/258] - Area Chart (ID: 122595)
Generating [10/258] - Scatter Plot (ID: 50003)
Generating [11/258] - Scatter Plot (ID: 118774)
Generating [12/258] - Box Plot (ID: 50419)
Generating [13/258] - Bar Chart (ID: 147731)
Generating [14/258] - Bar Chart (ID: 154422)
Generating [15/258] - Pie Chart (ID: 66936)
Generating [16/258] - Line Chart (ID: 56537)
Generating [17/258] - Scatter Plot (ID: 59868)
Generating [18/258] - Pie Chart (ID: 155348)
Generating [19/258] - Box Plot (ID: 105282)
Generating [20/258] - Line Chart (ID: 65440)
Generating [21/258] - Line Chart (ID: 150369)
Generating 

In [ ]:
# ==========================================
# PHASE 2: EVALUATION FROM SAVED FILES
# ==========================================
def run_evaluation():
    print(f"\nStarting Phase 2: Evaluating predictions from {PREDICTIONS_OUTPUT}...")
    
    with open(TEST_JSON_PATH, 'r', encoding="utf-8") as f:
        ground_truth_data = json.load(f)
        
    with open(PREDICTIONS_OUTPUT, 'r', encoding="utf-8") as f:
        predictions_data = json.load(f)

    # Convert ground truth to a dictionary for easy ID lookup
    gt_dict = {str(item["id"]): item for item in ground_truth_data if "id" in item}
    
    results = []

    for pred_item in predictions_data:
        chart_id = str(pred_item["id"])
        
        if chart_id not in gt_dict:
            continue
            
        gt_item = gt_dict[chart_id]
        
        # Extract the strings
        gt_spec_str = json.dumps(gt_item.get("ChartSpec", {}))
        pred_spec_str = pred_item["predicted_text"]
        
        # Run the rigorous evaluator
        metrics = evaluate_json_accuracy(gt_spec_str, pred_spec_str)
        
        # Attach metadata for the CSV
        metrics["chart_id"] = chart_id
        metrics["chart_type"] = pred_item.get("chart_type", "Unknown")
        metrics["complexity"] = pred_item.get("complexity", "Unknown")
        
        results.append(metrics)

    # Save to CSV
    df = pd.DataFrame(results)
    
    # Ensure math_accuracy handles None values gracefully
    df['math_accuracy'] = df['math_accuracy'].astype(float) 
    
    df.to_csv(METRICS_CSV, index=False)

    print("\n" + "="*60)
    print("🎉 EVALUATION COMPLETE")
    print("="*60)

    # Define the exact metrics we want to aggregate
    metric_cols = [
        'valid_json', 'schema_score', 'topology_score', 
        'domain_accuracy', 'numerical_accuracy', 
        'categorical_accuracy', 'math_accuracy', 'semantic_score'
    ]

    # 1. Group by chart_type
    summary = df.groupby('chart_type')[metric_cols].mean().reset_index()

    # 2. Calculate OVERALL averages across the entire test set
    overall = df[metric_cols].mean().to_frame().T
    overall['chart_type'] = 'OVERALL'
    
    # Combine chart types with the overall row
    summary = pd.concat([summary, overall], ignore_index=True)

    # Format all metric columns as clean percentages for the terminal
    for col in metric_cols:
        summary[col] = (summary[col] * 100).apply(lambda x: f"{x:.1f}%" if pd.notnull(x) else "N/A")
    
    # Print the primary summary table
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    print("📊 Performance by Chart Type:")
    print("-" * 60)
    print(summary.to_string(index=False))
    
    # 3. Print a secondary breakdown by complexity
    print("\n📊 Performance by Complexity:")
    print("-" * 60)
    comp_summary = df.groupby('complexity')[['valid_json', 'numerical_accuracy', 'schema_score', 'semantic_score']].mean().reset_index()
    for col in ['valid_json', 'numerical_accuracy', 'schema_score', 'semantic_score']:
        comp_summary[col] = (comp_summary[col] * 100).apply(lambda x: f"{x:.1f}%" if pd.notnull(x) else "N/A")
    print(comp_summary.to_string(index=False))

    print(f"\n📁 Full detailed metrics saved to: {METRICS_CSV}")

if __name__ == "__main__":
    run_evaluation()